## Background
This notebook provides the baseline for span identification task.

## Imports

In [12]:
import os
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from torch.utils.data import Dataset, random_split
from transformers import RobertaTokenizerFast, AutoModel, Trainer, TrainingArguments
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from safetensors.torch import load_file as safe_load_file

/Users/k_akhynko/Desktop/work/thesis/manipulative-narrative-detection/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
warnings.filterwarnings("ignore")

## Constants

In [14]:
TRAIN_PATH = "../../data/"
TRAIN_NAME = "train.parquet"
TEST_NAME = "test.csv"

MODEL_NAME = "FacebookAI/xlm-roberta-large"

In [5]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

Using device: cuda


## Read data

In [13]:
df = pd.read_parquet(os.path.join(TRAIN_PATH, TRAIN_NAME))
df.head()

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3822 entries, 0 to 3821
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             3822 non-null   object
 1   content        3822 non-null   object
 2   lang           3822 non-null   object
 3   manipulative   3822 non-null   bool  
 4   techniques     2589 non-null   object
 5   trigger_words  2589 non-null   object
dtypes: bool(1), object(5)
memory usage: 153.2+ KB


In [38]:
# If there are no manipulations in the text, we will set techniques and trigger_words to empty lists
df['techniques'] = df['techniques'].apply(lambda x: [] if x is None else x)
df['trigger_words'] = df['trigger_words'].apply(lambda x: [] if x is None else x)

## Dataset

In [9]:
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME, use_fast=True)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLMRobertaTokenizerFast'. 
The class this function is called from is 'RobertaTokenizerFast'.


In [20]:
class SpansDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.texts = df['content']
        self.spans = df['trigger_words']
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        spans = self.spans[idx]
        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=self.max_length, return_offsets_mapping=True)
        input_ids = encoding['input_ids']
        attention_mask = encoding['attention_mask']
        offset_mapping = encoding['offset_mapping']

        # Initialize token-level labels.
        # We'll mark tokens that are not real (i.e. padded tokens) as -100.
        token_labels = torch.full((self.max_length,), -100, dtype=torch.long)
        # For tokens that are not padding, set default label 0 (non-manipulative).
        for i in range(self.max_length):
            if attention_mask[i] == 1:
                token_labels[i] = 0

        # Loop over each token using its offset mapping.
        # If the token (defined by its character span) overlaps with any trigger span, label it as 1.
        # print(self.data.iloc[idx]['trigger_words_phrases'])
        for i, (token_start, token_end) in enumerate(offset_mapping):
            # Skip pad tokens
            if attention_mask[i] == 0:
                continue
            for span in spans:
                span_start, span_end = span
                # Check if there is any overlap between token span and trigger span.
                if token_end > span_start and token_start < span_end:
                    # print(text[token_start:token_end])
                    token_labels[i] = 1
                    break  # No need to check other spans for this token.

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'token_labels': torch.tensor(token_labels, dtype=torch.long)
            }

In [21]:
dataset = SpansDataset(df, tokenizer)

In [22]:
# Split dataset into training and validation sets
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

## Modeling

In [ ]:
class MultiTaskModel(nn.Module):
    def __init__(
        self,
        model_name="xlm-roberta-large",
        token_loss_weight=1.0,
    ):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, add_pooling_layer=False)
        hidden_size = self.encoder.config.hidden_size

        # Token classification head (binary classification with single output)
        self.token_classification_head = nn.Linear(hidden_size, 1)  # Single output for binary classification

        # Loss weights
        self.token_loss_weight = token_loss_weight

    def forward(self, input_ids, attention_mask, token_labels=None):
        # Get encoder outputs
        outputs = self.encoder(input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # (batch_size, seq_len, hidden_size)

        # Compute logits
        token_logits = self.token_classification_head(sequence_output)  # Token classification logits (batch_size, seq_len, 1)

        # Calculate losses if labels are provided
        loss = None
        if token_labels is not None:

            # Token classification loss (BCEWithLogitsLoss for binary classification)
            token_loss_fn = nn.BCEWithLogitsLoss()
            # Flatten predictions and labels
            flat_token_logits = token_logits.view(-1)
            flat_token_labels = token_labels.view(-1)

            # Create a mask for valid (non-pad) tokens
            active_mask = flat_token_labels != -100
            active_logits = flat_token_logits[active_mask]
            active_labels = flat_token_labels[active_mask].float()
            token_loss = token_loss_fn(active_logits, active_labels)

            # Weighted sum of both losses
            loss = self.token_loss_weight * token_loss

        return {
            "loss": loss,
            "logits": token_logits,
        }

In [ ]:
def get_f1_macro_thr_cv_stratified_token(targets, outputs, n_splits=5):
    """
    Compute the best threshold for binary classification using stratified K-fold cross-validation.
    """
    warnings.filterwarnings("ignore")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    thresholds = []
    metrics = []

    for train_idx, val_idx in skf.split(targets, targets):
        val_targets, val_outputs = targets[val_idx], outputs[val_idx]
        val_targets_flat = val_targets.flatten()
        val_outputs_flat = val_outputs.flatten()

        best_f1, best_thresh = 0, 0.01
        for threshold in np.arange(0.01, 0.99, 0.01):
            preds = (val_outputs > threshold).astype(int)
            f1 = f1_score(val_targets, preds)
            if f1 > best_f1:
                best_f1, best_thresh = f1, threshold

        thresholds.append(best_thresh)
        metrics.append(best_f1)

    avg_threshold = np.median(thresholds)
    targets_flat = targets.flatten()
    outputs_flat = outputs.flatten()
    validation_predictions = (outputs_flat > avg_threshold).astype(int)
    final_f1 = f1_score(targets_flat, validation_predictions)

    print("Stable averaged threshold across folds:", avg_threshold)
    print("Fold metrics:", metrics)
    print("Final F1-macro:", final_f1)

    return avg_threshold, final_f1


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # For single logit output (shape [batch, seq_len, 1])
    token_logits = logits.squeeze(-1)  # Remove the last dimension
    
    token_labels = labels
    valid_indices = token_labels != -100
    token_preds = torch.sigmoid(torch.tensor(token_logits)).cpu().numpy()

    _, token_f1 = get_f1_macro_thr_cv_stratified_token(token_labels[valid_indices], token_preds[valid_indices])

    return {
        "token_f1": token_f1
    }

In [29]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps = 200,
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    metric_for_best_model="token_f1",
    greater_is_better=True,
)

trainer = Trainer(
    model=MultiTaskModel(
        model_name=MODEL_NAME,
        token_loss_weight=1,
        dropout_prob=0.1,
    ),
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Step,Training Loss,Validation Loss,Token F1
200,No log,0.411461,0.606035
400,No log,0.431978,0.608576
600,0.462600,0.420802,0.615986
800,0.462600,0.484513,0.614631
1000,0.354100,0.472578,0.604943
1200,0.354100,0.631561,0.601459
1400,0.354100,0.525994,0.600107
1600,0.231900,0.650753,0.586237
1800,0.231900,0.633996,0.594215


Stable averaged threshold across folds: 0.3
Fold metrics: [0.6100051421435393, 0.6060399016124625, 0.6002279526998148, 0.6115420194976162, 0.6049382716049383]
Final F1-macro: 0.606034606034606
Stable averaged threshold across folds: 0.21000000000000002
Fold metrics: [0.6174394778781209, 0.6078790290489455, 0.604153041203401, 0.6131240864681898, 0.6054442575317327]
Final F1-macro: 0.6085761916522598
Stable averaged threshold across folds: 0.16
Fold metrics: [0.6229508196721312, 0.6120741183502689, 0.6140428026971563, 0.6190197512801756, 0.6126937269372694]
Final F1-macro: 0.6159864695933525
Stable averaged threshold across folds: 0.28
Fold metrics: [0.6230636833046471, 0.6102946297973114, 0.6140337932561056, 0.6185596560372626, 0.6135722347629797]
Final F1-macro: 0.6146309320880663
Stable averaged threshold across folds: 0.14
Fold metrics: [0.6170997754250882, 0.6024637793445418, 0.6058919044305265, 0.6033144222155867, 0.598462358224861]
Final F1-macro: 0.6049432487750419
Stable average

TrainOutput(global_step=1915, training_loss=0.30446536310659067, metrics={'train_runtime': 1799.6831, 'train_samples_per_second': 8.493, 'train_steps_per_second': 1.064, 'total_flos': 0.0, 'train_loss': 0.30446536310659067, 'epoch': 5.0})

In [30]:
trainer.evaluate()

Stable averaged threshold across folds: 0.04
Fold metrics: [0.6010483401281305, 0.5894659839063643, 0.5889244601866358, 0.590697016824544, 0.5861928104575164]
Final F1-macro: 0.5904325310921517


{'eval_loss': 0.6480362415313721,
 'eval_token_f1': 0.5904325310921517,
 'eval_runtime': 24.8472,
 'eval_samples_per_second': 30.788,
 'eval_steps_per_second': 3.864,
 'epoch': 5.0}

## Inference

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
MODEL_DUMP = "./results/checkpoint-766"
model = MultiTaskModel(
        model_name=MODEL_NAME,
        token_loss_weight=1,
        dropout_prob=0.1,
    )
# checkpoint_path = "checkpoint-768/model.safetensors"
state_dict = safe_load_file(MODEL_DUMP + "/model.safetensors")
model.load_state_dict(state_dict)
model.to(device)

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME, use_fast=True)

token_threshold = 0.16

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLMRobertaTokenizerFast'. 
The class this function is called from is 'RobertaTokenizerFast'.


In [ ]:
def run_inference(model, tokenizer, text, max_length=512, token_threshold=0.5, device=None):

    # Determine device if not provided
    if device is None:
        device = next(model.parameters()).device

    # Set model to evaluation mode
    model.eval()

    # Tokenize input text
    encoding = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
        return_offsets_mapping=True
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    offset_mapping = encoding['offset_mapping'][0].tolist()

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)

    # Process token classification head.
    # token_logits shape: (batch_size, seq_length, 1)
    token_logits = outputs["logits"].squeeze(-1)  # Now shape: (batch_size, seq_length)
    token_probs = torch.sigmoid(torch.tensor(token_logits)).cpu().numpy()
    token_preds = (token_probs > token_threshold)
    # Mask out padded tokens.
    token_preds = token_preds * attention_mask.cpu().numpy()

    # For span reconstruction, assume batch size 1.
    token_preds = token_preds[0].tolist()  # List of predictions per token.

    # Convert token predictions into spans using the offset mapping.
    spans = []
    current_span = None
    for pred, (tok_start, tok_end) in zip(token_preds, offset_mapping):
        # Skip tokens with dummy offsets (e.g., special tokens) if desired.
        if pred == 1:
            if current_span is None:
                current_span = [tok_start, tok_end]
            else:
                # Extend the span to the end of this token.
                current_span[1] = tok_end
        else:
            if current_span is not None:
                spans.append(tuple(current_span))
                current_span = None
    # Append any remaining span.
    if current_span is not None:
        spans.append(tuple(current_span))

    return {
        "token_probs": token_probs,
        "token_preds": token_preds,
        "spans": spans
    }


In [39]:
all_labels = set()
df['techniques'].apply(lambda x: all_labels.update(x))

0       None
1       None
2       None
3       None
4       None
        ... 
3817    None
3818    None
3819    None
3820    None
3821    None
Name: techniques, Length: 3822, dtype: object

In [15]:
test = pd.read_csv(os.path.join(TRAIN_PATH, TEST_NAME))

In [ ]:
# Predicting for test:
predictions = []
for text in tqdm(test.content.values):
    predictions.append(run_inference(model, tokenizer, text, device=device, token_threshold=token_threshold))

In [46]:
# Saving prediction for token:
pred_token = pd.DataFrame({
    "id": test.id.values
})

pred_values = [p["spans"] for p in predictions]
pred_token["trigger_words"] = pred_values

pred_token.to_csv("pred_token_baseline.csv", index=False)

In [ ]:
pred_token

,id,trigger_words
0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba,"[(0, 253)]"
1,9b2a61e4-d14e-4ff7-b304-e73d720319bf,"[(20, 26), (32, 52), (53, 74), (172, 188), (32..."
2,f0f1c236-80a8-4d25-b30c-a420a39be632,"[(0, 0), (20, 127), (142, 143)]"
3,31ea05ba-2c2b-4b84-aba7-f3cf6841b204,[]
4,a79e13ec-6d9a-40b5-b54c-7f4f743a7525,"[(0, 0), (24, 25), (87, 103), (127, 309)]"
...,...,...
5730,e8e22b6d-0068-4afb-b606-4a1baa8a8d4c,"[(0, 0), (32, 33), (113, 261), (367, 368), (42..."
5731,8b1d69b4-69ce-4e40-b4ba-dd2f370a8b6f,"[(0, 0), (3, 4), (15, 105), (174, 497)]"
5732,c2246217-3358-4f61-bda8-e2ec21aed5b2,"[(382, 480)]"
5733,45aa63c4-2248-4a0e-8f66-f3d23b6828ed,"[(0, 4), (67, 70), (259, 274)]"


## Calculate score

In [3]:
import ast
import pandas as pd

In [7]:
def span_f1(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Compute span-level F1 score based on overlap.

    Parameters:
    - solution (pd.DataFrame): Ground truth DataFrame with row ID and token labels.
    - submission (pd.DataFrame): Submission DataFrame with row ID and token labels.
    - row_id_column_name (str): Column name for the row identifier.

    Returns:
    - float: The token-level weighted F1 score.

    Example:
    >>> solution = pd.DataFrame({
    ...     "id": [1, 2, 3],
    ...     "trigger_words": [[(612, 622), (725, 831)], [(300, 312)], []]
    ... })
    >>> submission = pd.DataFrame({
    ...     "id": [1, 2, 3],
    ...     "trigger_words": [[(612, 622), (700, 720)], [(300, 312)], [(100, 200)]]
    ... })
    >>> score(solution, submission, "id")
    0.16296296296296295
    """
    if not all(col in solution.columns for col in ["id", "trigger_words"]):
        raise ValueError("Solution DataFrame must contain 'id' and 'trigger_words' columns.")
    if not all(col in submission.columns for col in ["id", "trigger_words"]):
        raise ValueError("Submission DataFrame must contain 'id' and 'trigger_words' columns.")
    
    def safe_parse_spans(trigger_words):
        if isinstance(trigger_words, str):
            try:
                return ast.literal_eval(trigger_words)
            except (ValueError, SyntaxError):
                return []
        if isinstance(trigger_words, (list, tuple)):
            return trigger_words
        return []

    def extract_tokens_from_spans(spans):
        tokens = set()
        for start, end in spans:
            tokens.update(range(start, end))
        return tokens
    
    solution = solution.copy()
    submission = submission.copy()

    solution["trigger_words"] = solution["trigger_words"].apply(safe_parse_spans)
    submission["trigger_words"] = submission["trigger_words"].apply(safe_parse_spans)

    merged = pd.merge(
        solution,
        submission,
        on="id",
        suffixes=("_solution", "_submission")
    )

    total_true_tokens = 0
    total_pred_tokens = 0
    overlapping_tokens = 0

    for _, row in merged.iterrows():
        true_spans = row["trigger_words_solution"]
        pred_spans = row["trigger_words_submission"]

        true_tokens = extract_tokens_from_spans(true_spans)
        pred_tokens = extract_tokens_from_spans(pred_spans)

        total_true_tokens += len(true_tokens)
        total_pred_tokens += len(pred_tokens)
        overlapping_tokens += len(true_tokens & pred_tokens)

    precision = overlapping_tokens / total_pred_tokens if total_pred_tokens > 0 else 0
    recall = overlapping_tokens / total_true_tokens if total_true_tokens > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return f1

In [18]:
baseline_span = pd.read_csv(os.path.join(TRAIN_PATH, "pred_span_xlm_roberta_large.csv"))
baseline_span['trigger_words'] = baseline_span['trigger_words'].apply(ast.literal_eval)

In [19]:
f1 = span_f1(baseline_span, test, row_id_column_name="id")
print("Span-level F1 score:", f1)

Span-level F1 score: 0.5858810450250138
